# nisar_tools end-to-end example

Out-of-core NISAR GSLC processing: crop → (merge) → interferograms → unwrap → mask → plot.

Every stage is lazy (dask) and persisted to a Zarr `Workspace`, so a full stack never needs to fit in memory, and an interrupted run resumes where it left off.

In [ ]:
from nisar_tools import GSLC, GSLCStack, InterferogramStack, UnwrappedStack, Workspace

# Same-frame, same-track granules across acquisition dates.
paths = [
    "/path/to/NISAR_..._20251128T023216_...h5",
    "/path/to/NISAR_..._20251210T023216_...h5",
]

# Geographic crop box (lon_min, lon_max, lat_min, lat_max).
bbox = (-118.5, -118.0, 34.0, 34.5)

ws = Workspace("workdir/")

## 1. Build and persist the cropped SLC stack

Persisting reopens the stack from Zarr, which severs the HDF5 file handles — after this the granules can be closed. The `ws.has(...)` check makes the cell resumable: a second run skips straight to loading.

In [ ]:
params = {"files": paths, "bbox": list(bbox)}

if ws.has("slc_stack", {"stage": "slc_stack", "epsg": None, **params}):
    stack = GSLCStack.from_zarr(ws.path("slc_stack"))
else:
    gslcs = [GSLC(p) for p in paths]
    stack = GSLCStack.from_gslcs(gslcs, bbox=bbox).persist(ws, "slc_stack", **params)
    for g in gslcs:
        g.close()

stack

### Optional: merge an adjacent frame in the same track

Build a second `GSLCStack` from the neighbouring frame (same dates) and merge onto the union grid. Only do this for frames in the same track; to combine different tracks, merge the individually unwrapped interferograms instead.

```python
other = GSLCStack.from_gslcs([GSLC(p) for p in other_frame_paths], bbox=bbox)
stack = stack.merge(other)
```

## 2. Form multilooked interferograms

In [ ]:
igrams = stack.form_interferograms(
    pairs="sequential",   # or "all", or [(0, 1), (0, 2), ...]
    looks=5,
    downsample=True,
    convolution="Uniform",
).persist(ws, "igrams")

fig, ax = igrams.plot_wrapped(pair=0)

## 3. Unwrap with SNAPHU

Unwrapping runs one pair at a time, writing each result into its own region of the output store and flagging it done. Peak memory is one pair, and a re-run resumes at the first unfinished pair.

In [ ]:
unw = igrams.unwrap(ws, nproc=8)
unw

## 4. Mask water and plot

In [ ]:
unw = unw.mask_water(workspace=ws)   # requires the optional pygmt/GMT install
fig, ax = unw.plot(pair=0)

# Reproject a single pair to lon/lat for export.
unw_latlon = unw.to_latlon(pair=0)
# unw_latlon.rio.to_raster("unwrapped_pair0.tif")